# ARTICLE2_NB6: Literature audit (RQ1)

**This notebook produces results for the fault-injection study (see README).**

Protocol: *study protocol* (frozen 11 Nov 2026), Section 7. The audit reports **missing safeguards**, never proven errors in other people's work. In the text, discuss practices, not named papers.

**Run it in stages.** Each stage reads what you filled in and tells you the next step. Re-running a stage never overwrites a sheet you have started filling in.

| Stage | You do | This notebook does |
|---|---|---|
| 1 | Run the searches; put the exports in `06_Literature_Audit/01_search_exports`; enter the search log in CELL 2 | Reads the Scopus and Lens.org exports, removes duplicates, runs the Google Scholar cross-check, writes the screening sheet |
| 2 | Screen titles and abstracts in `02_screening/screening_sheet.xlsx` | Writes the full-text sheet for every Include or Unsure |
| 3 | Check full texts in `03_fulltext/fulltext_check.xlsx` | Numbers the eligible set, draws 40 with seed 2026, writes the coding sheets |
| 4 | Code the 40; at least 7 days later re-code the 10 without looking | Field counts, Cohen's kappa per field, safeguards vs claimed improvement, flow diagram |

**Inputs and outputs:** reads and writes only `Article 2/06_Literature_Audit` (plus a copy of the audit figure in `05_Outputs/figures`). No GPU needed.

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
print("Working directory:", ROOT)

## CELL 1: Settings, folders and helpers

In [ ]:
import re, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, Alignment, PatternFill
from sklearn.metrics import cohen_kappa_score
from scipy import stats

LA = ROOT / "06_Literature_Audit"
D_EXP, D_SCR, D_FT, D_COD, D_RES = (LA / "01_search_exports", LA / "02_screening", LA / "03_fulltext",
                                    LA / "04_coding", LA / "05_results")
for d in [D_EXP, D_SCR, D_FT, D_COD, D_RES]:
    d.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "05_Outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

AUDIT_SEED = 2026          # Protocol Section 7, fixed 11 Nov 2026 before any search
SAMPLE_SIZE = 40
RECODE_SIZE = 10
RECODE_MIN_DAYS = 7

SEARCH_STRING = ('(hybrid OR ensemble OR combin*) AND (ARIMA OR ARFIMA OR GARCH OR "exponential smoothing") AND '
                 '(LSTM OR GRU OR XGBoost OR "random forest" OR "neural network" OR "deep learning" OR "machine learning") AND '
                 '("stock price" OR "stock market" OR "stock index" OR "share price" OR equit*) AND (forecast* OR predict*)')

EXCLUSION_REASONS = ["Not stock or stock-index prices/returns", "Cryptocurrency, FX or commodity only", "No statistical + ML hybrid",
                     "Volatility-only target", "Not daily (or intraday aggregated to daily)", "No out-of-sample accuracy reported",
                     "Review", "Conference paper", "Not in English", "Published before 2022", "Full text not available", "Other (note)"]
FIELDS = {  # Protocol Section 7 coding sheet: field -> allowed values
    "Persistence benchmark": ["Yes", "No", "Unclear"],
    "Relative or returns metric": ["Yes", "No", "Unclear"],
    "Price R2 as headline": ["Yes", "No"],
    "Significance test": ["Yes", "No"],
    "Robust variance": ["Yes", "No", "Not applicable"],
    "Multiple-testing control": ["Yes", "No", "Not applicable"],
    "Chronological split": ["Yes", "No", "Unclear"],
    "Leakage control stated": ["Yes", "No"],
    "Same horizon for all components": ["Yes", "No", "Unclear"],
    "Unit consistency stated": ["Yes", "No", "Unclear"],
    "Code public": ["Yes", "No"],
    "Data public": ["Yes", "No"],
    "Headline improvement": None,     # a number (%) or "Not given"
}
SAFEGUARDS = [f for f in FIELDS if f not in ("Price R2 as headline", "Headline improvement")]

def norm_doi(x):
    x = str(x).strip().lower() if pd.notna(x) else ""
    return re.sub(r"^(https?://(dx\.)?doi\.org/|doi:)", "", x)

def norm_title(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower()) if pd.notna(x) else ""

def write_sheet(df, path, dropdowns, widths=None, wrap=()):
    """Write an Excel sheet with drop-down lists. Never overwrites an existing file."""
    if path.exists():
        print(f"{path.name} already exists - not overwritten (delete it yourself to rebuild).")
        return False
    df.to_excel(path, index=False)
    wb = load_workbook(path); ws = wb.active
    cols = {c.value: c.column_letter for c in ws[1]}
    for c in ws[1]:
        c.font = Font(bold=True); c.fill = PatternFill("solid", fgColor="F0EFEC")
    n = ws.max_row
    for col, values in dropdowns.items():
        dv = DataValidation(type="list", formula1='"' + ",".join(values) + '"', allow_blank=True)
        ws.add_data_validation(dv); dv.add(f"{cols[col]}2:{cols[col]}{max(n, 2)}")
    for col, w in (widths or {}).items():
        ws.column_dimensions[cols[col]].width = w
    for col in wrap:
        for cell in ws[cols[col]][1:]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "B2"
    wb.save(path)
    print("Written:", path.relative_to(ROOT))
    return True
print("Ready. Audit seed:", AUDIT_SEED)

## CELL 2: Stage 1 - read the exports, remove duplicates, write the screening sheet

**Before running:** run the searches (see the instructions file in `06_Literature_Audit`), save the exports in `01_search_exports`, and fill in `SEARCH_LOG` below with what each database showed on screen.
- Scopus: CSV export with all fields (including abstract). File name must start with `scopus`.
- Lens.org (replaces Web of Science, which Taylor's does not subscribe to): CSV export. File name must start with `lens`.
- Google Scholar cross-check (optional, recall only): Publish or Perish CSV. File name must start with `gscholar`.
- Web of Science exports (`wos*`) are still read if they ever become available.

In [ ]:
SEARCH_LOG = [
    # {"Database": "Scopus", "Date": "2026-11-12", "Hits_shown": 0, "Query": "TITLE-ABS-KEY(...) AND PUBYEAR > 2021 AND DOCTYPE(ar) AND SRCTYPE(j) AND LANGUAGE(english)"},
    # {"Database": "Lens.org", "Date": "2026-11-12", "Hits_shown": 0, "Query": "(title/abstract/keyword per concept group) ; Journal Article ; 2022-2026"},
    # Web of Science: not available through Taylor's Library (deviation log, 12 Nov 2026)
]

def read_scopus(p):
    df = pd.read_csv(p, encoding="utf-8-sig")
    g = lambda c: df[c] if c in df else pd.Series([np.nan] * len(df))
    return pd.DataFrame({"Title": g("Title"), "Abstract": g("Abstract"), "Year": g("Year"), "Journal": g("Source title"),
                         "DOI": g("DOI"), "Authors": g("Authors"), "Doc_type": g("Document Type"),
                         "Language": g("Language of Original Document"), "Source_db": "Scopus"})

def read_wos(p):
    if p.suffix.lower() in (".xls", ".xlsx"):
        df = pd.read_excel(p)
    else:
        df = pd.read_csv(p, sep="\t", quoting=3, index_col=False, encoding="utf-8-sig", dtype=str)
    pick = lambda *names: next((df[n] for n in names if n in df), pd.Series([np.nan] * len(df)))
    return pd.DataFrame({"Title": pick("Article Title", "TI"), "Abstract": pick("Abstract", "AB"),
                         "Year": pick("Publication Year", "PY"), "Journal": pick("Source Title", "SO"), "DOI": pick("DOI", "DI"),
                         "Authors": pick("Authors", "AU"), "Doc_type": pick("Document Type", "DT"), "Language": pick("Language", "LA"),
                         "Source_db": "Web of Science"})

assert SEARCH_LOG, "Fill in SEARCH_LOG (database, date, hits shown, exact query) before running this cell."
def read_lens(p):
    df = pd.read_csv(p, encoding="utf-8-sig", dtype=str)
    pick = lambda *names: next((df[n] for n in names if n in df), pd.Series([np.nan] * len(df)))
    return pd.DataFrame({"Title": pick("Title"), "Abstract": pick("Abstract"), "Year": pick("Publication Year", "Year Published"),
                         "Journal": pick("Source Title"), "DOI": pick("DOI"), "Authors": pick("Author/s", "Authors"),
                         "Doc_type": pick("Publication Type"), "Language": pick("Language"), "Source_db": "Lens.org"})

def read_pop(p):
    """Google Scholar results exported with Publish or Perish (cross-check only; not added to the record set)."""
    df = pd.read_csv(p, encoding="utf-8-sig", dtype=str)
    pick = lambda *names: next((df[n] for n in names if n in df), pd.Series([np.nan] * len(df)))
    return pd.DataFrame({"Title": pick("Title"), "Year": pd.to_numeric(pick("Year"), errors="coerce"),
                         "Source": pick("Source"), "DOI": pick("DOI"), "Rank": pick("GSRank")})

files = sorted(p for p in D_EXP.iterdir() if p.is_file() and not p.name.startswith("."))
sc = [read_scopus(p) for p in files if p.name.lower().startswith("scopus")]
wo = [read_wos(p) for p in files if p.name.lower().startswith("wos")]
le = [read_lens(p) for p in files if p.name.lower().startswith("lens")]
gs = [read_pop(p) for p in files if p.name.lower().startswith("gscholar")]
assert sc or wo or le, f"No exports found in {D_EXP}. Name them scopus*.csv, lens*.csv (or wos*.xls / wos*.txt)"
recs = pd.concat(sc + wo + le, ignore_index=True)        # Scopus first, so its record is kept for duplicates
recs["Year"] = pd.to_numeric(recs["Year"], errors="coerce")
recs["doi_n"], recs["title_n"] = recs.DOI.map(norm_doi), recs.Title.map(norm_title)
recs["key"] = np.where(recs.doi_n != "", "doi:" + recs.doi_n, "t:" + recs.title_n + ":" + recs.Year.astype(str))
counts = recs.groupby("Source_db").size().to_dict()
dbs = recs.groupby("key").Source_db.agg(lambda s: "; ".join(sorted(set(s))))
# a record without DOI can still duplicate one with DOI: second pass on normalised title + year
dedup = recs.drop_duplicates("key").copy()
dedup["Found_in"] = dedup.key.map(dbs)
dedup["ty"] = dedup.title_n + ":" + dedup.Year.astype(str)
n_before_title = len(dedup)
dedup = dedup.sort_values("doi_n", ascending=False).drop_duplicates("ty").sort_values(["Year", "title_n"])
dedup["ID"] = [f"A{i:04d}" for i in range(1, len(dedup) + 1)]

flow = {"records_" + k.replace(" ", "_"): int(v) for k, v in counts.items()}
flow["records_total"] = int(len(recs))
flow["duplicates_removed"] = int(len(recs) - len(dedup))
flow["records_screened"] = int(len(dedup))
log = pd.DataFrame(SEARCH_LOG)
if not log.empty:
    log["Records_in_export"] = log.Database.map(counts)
    log["Matches_export"] = log.Hits_shown == log.Records_in_export
log.to_csv(D_RES / "search_log.csv", index=False)
json.dump(flow, open(D_RES / "flow_counts_stage1.json", "w"), indent=2)
print("Search log:"); print(log.to_string(index=False) if not log.empty else "  (empty - fill in SEARCH_LOG above)")
print("Flow so far:", flow)

if gs:   # Google Scholar recall cross-check (Protocol Section 7); reported, records not added
    g = pd.concat(gs, ignore_index=True)
    g = g[g.Year >= 2022].copy()
    g["found"] = g.DOI.map(norm_doi).isin(set(dedup.doi_n) - {""}) | g.Title.map(norm_title).isin(set(dedup.title_n))
    g.to_csv(D_RES / "gscholar_crosscheck.csv", index=False)
    flow["gscholar_checked"], flow["gscholar_found"] = int(len(g)), int(g.found.sum())
    json.dump(flow, open(D_RES / "flow_counts_stage1.json", "w"), indent=2)
    print(f"Google Scholar cross-check: {int(g.found.sum())} of {len(g)} results (2022 on) are already in the record set "
          f"({100 * g.found.mean():.0f}%). Not found: see 05_results/gscholar_crosscheck.csv")

scr = dedup[["ID", "Title", "Year", "Journal", "Authors", "DOI", "Found_in", "Doc_type", "Abstract"]].copy()
scr["Decision"], scr["Reason"], scr["Note"] = "", "", ""
write_sheet(scr, D_SCR / "screening_sheet.xlsx", {"Decision": ["Include", "Exclude", "Unsure"], "Reason": EXCLUSION_REASONS},
            widths={"Title": 60, "Abstract": 90, "Reason": 30, "Note": 30}, wrap=["Title", "Abstract"])
print()
print("NEXT: screen titles and abstracts in 02_screening/screening_sheet.xlsx (Decision; Reason for every Exclude), then run CELL 3.")

## CELL 3: Stage 2 - read the screening decisions, write the full-text sheet

Every **Include** and **Unsure** goes to the full-text check. Every **Exclude** needs a reason.

In [ ]:
scr = pd.read_excel(D_SCR / "screening_sheet.xlsx", dtype={"ID": str})
blank = scr[scr.Decision.isna() | (scr.Decision.astype(str).str.strip() == "")]
assert blank.empty, f"{len(blank)} records have no Decision yet (first: {blank.ID.head(5).tolist()})"
bad = scr[~scr.Decision.isin(["Include", "Exclude", "Unsure"])]
assert bad.empty, f"Unknown decisions: {bad.Decision.unique().tolist()}"
noreason = scr[(scr.Decision == "Exclude") & scr.Reason.isna()]
assert noreason.empty, f"{len(noreason)} exclusions without a reason (first: {noreason.ID.head(5).tolist()})"
ta_excl = scr[scr.Decision == "Exclude"].Reason.value_counts()
to_ft = scr[scr.Decision.isin(["Include", "Unsure"])][["ID", "Title", "Year", "Journal", "Authors", "DOI"]].copy()
to_ft["Full_text_obtained"], to_ft["Eligible"], to_ft["Reason"], to_ft["Note"] = "", "", "", ""
print("Title/abstract screening:", scr.Decision.value_counts().to_dict())
print("Exclusion reasons:"); print(ta_excl.to_string())
ta_excl.to_csv(D_RES / "exclusions_title_abstract.csv")
write_sheet(to_ft, D_FT / "fulltext_check.xlsx",
            {"Full_text_obtained": ["Yes", "No"], "Eligible": ["Yes", "No"], "Reason": EXCLUSION_REASONS},
            widths={"Title": 60, "Reason": 30, "Note": 30}, wrap=["Title"])
print()
print(f"NEXT: check the full text of all {len(to_ft)} records in 03_fulltext/fulltext_check.xlsx, then run CELL 4.")

## CELL 4: Stage 3 - number the eligible set, draw 40 with seed 2026, write the coding sheets

Protocol order: every remaining full text is checked, the eligible set is numbered (by ID), then 40 are drawn at random with seed 2026; if fewer than 40 are eligible, all are coded. The 10 for re-coding are then drawn from the 40 with the same generator (deviation log, 12 Nov 2026).

In [ ]:
ft = pd.read_excel(D_FT / "fulltext_check.xlsx", dtype={"ID": str})
blank = ft[ft.Eligible.isna()]
assert blank.empty, f"{len(blank)} full texts not yet decided (first: {blank.ID.head(5).tolist()})"
noreason = ft[(ft.Eligible == "No") & ft.Reason.isna()]
assert noreason.empty, f"{len(noreason)} full-text exclusions without a reason"
eligible = ft[ft.Eligible == "Yes"].sort_values("ID").reset_index(drop=True)
rng = np.random.default_rng(AUDIT_SEED)
k = min(SAMPLE_SIZE, len(eligible))
sample_idx = np.sort(rng.choice(len(eligible), size=k, replace=False)) if len(eligible) > SAMPLE_SIZE else np.arange(len(eligible))
sample = eligible.iloc[sample_idx].reset_index(drop=True)
recode_ids = sorted(rng.choice(sample.ID.values, size=min(RECODE_SIZE, k), replace=False).tolist())
flow3 = {"fulltext_assessed": int(len(ft)), "fulltext_excluded": int((ft.Eligible == "No").sum()),
         "eligible": int(len(eligible)), "sampled": int(k), "recode": len(recode_ids)}
json.dump(flow3, open(D_RES / "flow_counts_stage3.json", "w"), indent=2)
ft[ft.Eligible == "No"].Reason.value_counts().to_csv(D_RES / "exclusions_fulltext.csv")
sample.to_csv(D_RES / "audit_sample_40.csv", index=False)
pd.Series(recode_ids, name="ID").to_csv(D_RES / "recode_ids.csv", index=False)
print("Flow:", flow3)
print("Re-code IDs (do NOT open the first codes when re-coding):", recode_ids)

def coding_frame(df):
    out = df[["ID", "Authors", "Year", "Title", "Journal", "DOI"]].copy()
    for f in FIELDS:
        out[f] = ""
    out["Headline improvement source (quote location)"] = ""
    out["Coder"], out["Date coded"], out["Notes"] = "", "", ""
    return out
dd = {f: v for f, v in FIELDS.items() if v}
first = write_sheet(coding_frame(sample), D_COD / "coding_sheet_first.xlsx", dd, widths={"Title": 50, "Notes": 40}, wrap=["Title"])
recode = write_sheet(coding_frame(sample[sample.ID.isin(recode_ids)]), D_COD / "coding_sheet_recode.xlsx", dd,
                     widths={"Title": 50, "Notes": 40}, wrap=["Title"])
print()
print("NEXT: code the papers in 04_coding/coding_sheet_first.xlsx (fill 'Date coded').")
print(f"At least {RECODE_MIN_DAYS} days later, fill 04_coding/coding_sheet_recode.xlsx WITHOUT opening the first sheet; then run CELL 5.")

## CELL 5: Stage 4 - results: field counts, Cohen's kappa, safeguards vs claimed improvement, flow diagram

Descriptive only. The safeguard score counts the eleven fields where "Yes" is a safeguard, plus one point when price R2 is *not* the headline.

In [ ]:
first = pd.read_excel(D_COD / "coding_sheet_first.xlsx", dtype={"ID": str})
rec = pd.read_excel(D_COD / "coding_sheet_recode.xlsx", dtype={"ID": str})
problems = []
for name, df in [("first", first), ("recode", rec)]:
    for f, allowed in FIELDS.items():
        if allowed:
            badv = df[~df[f].isin(allowed)]
            problems += [f"{name} {r.ID}: {f} = {r[f]!r}" for _, r in badv.iterrows()]
        else:
            ok = df[f].astype(str).str.strip().str.lower().eq("not given") | pd.to_numeric(df[f], errors="coerce").notna()
            problems += [f"{name} {r.ID}: {f} = {r[f]!r} (number or 'Not given')" for _, r in df[~ok].iterrows()]
assert not problems, "Fix these codes first:\n" + "\n".join(problems[:30])
d1 = pd.to_datetime(first.set_index("ID")["Date coded"]).reindex(rec.ID)
d2 = pd.to_datetime(rec.set_index("ID")["Date coded"])
gap = (d2 - d1).dt.days
assert (gap >= RECODE_MIN_DAYS).all(), f"Re-coding must be at least {RECODE_MIN_DAYS} days after first coding: {gap.to_dict()}"

# Field counts (Protocol: counts and percentages per field)
rows = []
for f, allowed in FIELDS.items():
    if allowed:
        vc = first[f].value_counts()
        for v in allowed:
            rows.append({"Field": f, "Value": v, "Count": int(vc.get(v, 0)), "Percent": round(100 * vc.get(v, 0) / len(first), 1)})
counts = pd.DataFrame(rows)
counts.to_csv(D_RES / "audit_field_counts.csv", index=False)
print(f"Field counts (n = {len(first)}):")
print(counts.pivot(index="Field", columns="Value", values="Count").reindex(list(f for f in FIELDS if FIELDS[f])).fillna(0).astype(int).to_string())

# Reliability: Cohen's kappa per field (first vs re-code, same papers)
m = first.set_index("ID").loc[rec.ID]
kap = []
for f, allowed in FIELDS.items():
    a, b = m[f].astype(str).str.strip().values, rec.set_index("ID").loc[rec.ID][f].astype(str).str.strip().values
    agree = float(np.mean(a == b))
    k = cohen_kappa_score(a, b) if len(set(a) | set(b)) > 1 else np.nan
    kap.append({"Field": f, "Percent_agreement": round(100 * agree, 1), "Cohen_kappa": None if np.isnan(k) else round(k, 3)})
kap = pd.DataFrame(kap)
kap.to_csv(D_RES / "audit_kappa.csv", index=False)
print()
print("Reliability (first vs re-code; kappa undefined when only one category was used):")
print(kap.to_string(index=False))

# Safeguards vs claimed improvement (descriptive)
score = sum((first[f] == "Yes").astype(int) for f in SAFEGUARDS) + (first["Price R2 as headline"] == "No").astype(int)
claim = pd.to_numeric(first["Headline improvement"], errors="coerce")
sv = pd.DataFrame({"ID": first.ID, "Safeguard_score": score, "Headline_improvement_pct": claim})
sv.to_csv(D_RES / "audit_safeguards_vs_claim.csv", index=False)
ok = sv.dropna()
rho = stats.spearmanr(ok.Safeguard_score, ok.Headline_improvement_pct) if len(ok) >= 5 else None
print()
print(f"Safeguard score (0-{len(SAFEGUARDS) + 1}): median {score.median():.0f}, range {score.min()}-{score.max()}")
print(f"Papers with a numeric headline improvement: {len(ok)} of {len(first)}")
if rho is not None:
    print(f"Spearman rho (descriptive only): {rho.statistic:.3f}")

# Flow diagram (counts from stages 1 and 3)
f1 = json.load(open(D_RES / "flow_counts_stage1.json")); f3 = json.load(open(D_RES / "flow_counts_stage3.json"))
scr = pd.read_excel(D_SCR / "screening_sheet.xlsx")
dbs_counts = {k.replace("records_", "").replace("_", " "): v for k, v in f1.items()
              if k.startswith("records_") and k not in ("records_total", "records_screened")}
boxes = ["Records identified\n" + "; ".join(f"{k} {v}" for k, v in dbs_counts.items()),
         f"After duplicates removed\nn = {f1['records_screened']}",
         f"Titles and abstracts screened\nexcluded: {(scr.Decision == 'Exclude').sum()}",
         f"Full texts assessed\nn = {f3['fulltext_assessed']}; excluded: {f3['fulltext_excluded']}",
         f"Eligible studies\nn = {f3['eligible']}",
         f"Random sample coded (seed {AUDIT_SEED})\nn = {f3['sampled']}; re-coded: {f3['recode']}"]
fig, ax = plt.subplots(figsize=(4.4, 6.0)); ax.axis("off")
ys = [0.95 - i * 0.18 for i in range(len(boxes))]
for i, (t, y) in enumerate(zip(boxes, ys)):
    ax.text(0.5, y, t, ha="center", va="center", fontsize=7.5, transform=ax.transAxes,
            bbox=dict(boxstyle="round,pad=0.5", fc="#f0efec", ec="#52514e", lw=0.6))
    if i < len(boxes) - 1:
        ax.annotate("", xy=(0.5, ys[i + 1] + 0.055), xytext=(0.5, y - 0.055), xycoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="#52514e", lw=0.8))
for p in [D_RES / "audit_flow_diagram.png", FIG / "Figure_7_audit_flow_diagram.png"]:
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# Supplementary table: the audited papers, cited as a group
sup = first[["ID", "Authors", "Year", "Title", "Journal", "DOI"]]
sup.to_csv(D_RES / "Table_S7_audited_papers.csv", index=False)
(ROOT / "05_Outputs" / "logs").mkdir(parents=True, exist_ok=True)
json.dump({"notebook": "ARTICLE2_NB6_Literature_Audit", "purpose": "Article 2 only", "run_at": datetime.now().isoformat(timespec="seconds"),
           "seed": AUDIT_SEED, "flow": {**f1, **f3}, "n_coded": int(len(first)), "n_recoded": int(len(rec))},
          open(ROOT / "05_Outputs" / "logs" / "ARTICLE2_NB6_run_log.json", "w"), indent=2)
print("Done. Results in 06_Literature_Audit/05_results. Please share the output of this cell.")